# Part 2: Computer Vision — Manufacturing Defect Classification
**Goal:** Build a CNN to classify product surface images into 4 defect categories.

## Setup: Upload Dataset
Upload the zip file when prompted, or mount Google Drive.

In [ ]:
# Upload the part 2 dataset zip or extract from Drive
from google.colab import files
import zipfile, os

uploaded = files.upload()  # Upload the zip file here

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

# Find the part 2 images folder
for root, dirs, files_ in os.walk('.'):
    if 'part_2_cnn_computer_vision' in root and 'images' in dirs:
        BASE_DIR = os.path.join(root, 'images')
        print('Found dataset at:', BASE_DIR)
        break

In [ ]:
# If you already extracted, just set this manually:
# BASE_DIR = 'ai_project_synthetic_datasets/part_2_cnn_computer_vision/images'

import os
os.makedirs('results', exist_ok=True)
os.makedirs('sample_predictions', exist_ok=True)
print('Folders ready.')

---
## Task 1: Problem Identification

**Problem Type: Image Classification**

Each image shows a product surface and needs to be assigned exactly one label from:
`normal`, `scratch`, `dent`, `stain`.

Since we're assigning a single class label per image (not detecting bounding boxes or segmenting pixel regions),
this is a **multi-class image classification** problem — the most appropriate formulation.

**Real-world context:** In manufacturing, automated visual inspection systems use CNNs to flag defective
products on an assembly line in real-time, replacing manual human inspection.

---
## Task 2: Dataset Exploration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import glob

CLASSES = ['dent', 'normal', 'scratch', 'stain']

print('CLASS DISTRIBUTION')
print('-' * 30)
total = 0
for cls in CLASSES:
    imgs = glob.glob(f'{BASE_DIR}/{cls}/*.png')
    print(f'  {cls:<10}: {len(imgs)} images')
    total += len(imgs)
print(f'  {"TOTAL":<10}: {total} images')

# Check image dimensions
sample_path = glob.glob(f'{BASE_DIR}/normal/*.png')[0]
img = Image.open(sample_path)
print(f'\nImage size : {img.size[0]}x{img.size[1]} px')
print(f'Color mode : {img.mode}')
print('\nDataset is perfectly balanced — no class imbalance issue.')

In [ ]:
# Show sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for col, cls in enumerate(CLASSES):
    imgs = sorted(glob.glob(f'{BASE_DIR}/{cls}/*.png'))
    for row in range(2):
        ax = axes[row][col]
        img = Image.open(imgs[row])
        ax.imshow(img)
        ax.set_title(cls.upper(), fontweight='bold', fontsize=11)
        ax.axis('off')

plt.suptitle('Sample Images — One Per Class (2 Samples Each)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Task 3: Image Preprocessing

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

IMG_SIZE = 64   # resize to 64x64
BATCH_SIZE = 32

# Load all images and labels
X, y = [], []
label_map = {cls: i for i, cls in enumerate(CLASSES)}

for cls in CLASSES:
    for path in sorted(glob.glob(f'{BASE_DIR}/{cls}/*.png')):
        img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        X.append(np.array(img))
        y.append(label_map[cls])

X = np.array(X, dtype='float32') / 255.0   # normalize to [0, 1]
y = np.array(y)

print(f'X shape : {X.shape}   (images x height x width x channels)')
print(f'y shape : {y.shape}')
print(f'Pixel range: [{X.min():.2f}, {X.max():.2f}]')

In [ ]:
# Train/Val/Test split: 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Training   : {X_train.shape[0]} images')
print(f'Validation : {X_val.shape[0]} images')
print(f'Testing    : {X_test.shape[0]} images')

# Data augmentation on training set
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
], name='augmentation')

print('\nAugmentation applied: horizontal flip, rotation, zoom')

---
## Task 4: CNN Model Creation

In [ ]:
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                      Dense, Dropout, BatchNormalization)

model = Sequential([
    Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    data_augmentation,

    # Block 1
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(2, 2),

    # Block 2
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(2, 2),

    # Block 3
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(2, 2),

    # Classifier head
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(4, activation='softmax')   # 4 classes
], name='DefectCNN')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

**Architecture walkthrough:**
- **3 Conv blocks**: Each block learns progressively complex features (edges → textures → patterns), followed by MaxPooling to reduce spatial size
- **Flatten**: Converts 3D feature maps into a 1D vector
- **Dense(128) + Dropout(0.4)**: Learns high-level combinations; Dropout prevents overfitting
- **Dense(4, softmax)**: Outputs probabilities across 4 classes

---
## Task 5: Model Training and Evaluation

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['loss'], label='Train', color='#3498db', lw=2)
axes[0].plot(history.history['val_loss'], label='Validation', color='#e74c3c', lw=2, ls='--')
axes[0].set_title('Loss Over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Train', color='#2ecc71', lw=2)
axes[1].plot(history.history['val_accuracy'], label='Validation', color='#e67e22', lw=2, ls='--')
axes[1].set_title('Accuracy Over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/accuracy_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/accuracy_loss_curves.png')

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Test set evaluation
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')

# Confusion matrix
y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/confusion_matrix.png')

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=CLASSES))

In [ ]:
# Sample predictions on test images
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

indices = np.random.choice(len(X_test), 8, replace=False)
for i, idx in enumerate(indices):
    img = X_test[idx]
    actual = CLASSES[y_test[idx]]
    predicted = CLASSES[y_pred[idx]]
    correct = actual == predicted

    axes[i].imshow(img)
    color = '#2ecc71' if correct else '#e74c3c'
    axes[i].set_title(f'Actual: {actual}\nPred: {predicted}',
                      color=color, fontsize=9, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_predictions/prediction_outputs.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sample_predictions/prediction_outputs.png')

In [ ]:
# Download all result files
from google.colab import files
files.download('results/accuracy_loss_curves.png')
files.download('results/confusion_matrix.png')
files.download('sample_predictions/prediction_outputs.png')